# Advanced PyTorch: torch.compile and Friends

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/05-pytorch/07-Advanced-PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Recommended environment: Google Colab with a T4 GPU (`Runtime → Change runtime type → T4 GPU`).

This notebook is mostly **explanation**: how `torch.compile` works, what can go wrong, and a few other advanced tools you will meet in modern PyTorch code. The code cells are small demonstrations, not experiments; they run on a T4, a TPU or even a CPU.

Contents
1. Eager mode vs. graphs: why compile?
2. How `torch.compile` works: Dynamo, AOTAutograd, Inductor
3. Guards and recompilation, dynamic shapes
4. Graph breaks
5. Compile options: `mode`, `fullgraph`, `backend`, regional compilation, caching
6. `torch.export`: capturing a model for deployment
7. `torch.func`: function transforms (`grad`, `vmap`)
8. Saving memory: activation checkpointing
9. Finding the bottleneck: `torch.profiler`
10. FlexAttention: custom attention written in Python, compiled to a fused kernel

In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(torch.__version__, device)

## 1. Eager mode vs. graphs: why compile?

Everything we did so far ran in **eager mode**: Python executes your code line by line, and each operation (`a + b`, `x @ W`, `relu`, ...) immediately launches one kernel on the GPU. This is why PyTorch feels like NumPy and is easy to debug: you can `print` any tensor at any point.

The price:

* **Python overhead**: every operation goes through the Python interpreter and the PyTorch dispatcher, a few microseconds each. For small tensors the GPU finishes faster than Python can send the next operation, and sits idle.
* **No fusion**: `y = torch.relu(x * 2 + 1)` launches 3 kernels, each reading its input from GPU memory and writing its output back. Memory traffic, not arithmetic, is often the real limit. A single fused kernel would read `x` once and write `y` once.
* **No global view**: each operation is optimized on its own, never as part of the whole computation.

A **compiler** that sees the whole computation as a **graph** can fuse operations, remove Python overhead and choose better kernels. `torch.compile` (PyTorch 2.0+) does this *without* asking you to rewrite your code in a different style: you keep writing normal PyTorch.

```python
model = torch.compile(model)          # or decorate a function with @torch.compile
```

## 2. How `torch.compile` works

```
your Python code
   │  TorchDynamo   — reads the Python *bytecode* while the function runs, records the PyTorch
   │                  operations into an FX graph, and installs "guards"
   ▼
FX graph (forward)
   │  AOTAutograd   — traces the backward pass ahead of time, producing forward + backward graphs,
   │                  and decomposes big ops into ~250 primitive ops
   ▼
forward / backward graphs of primitive ops
   │  backend       — the compiler that turns a graph into code
   │                  "inductor" (default): generates Triton kernels for GPUs, C++/OpenMP for CPUs
   │                  "tpu": XLA, for TPUs (see 06)
   ▼
fast compiled code, cached and reused
```

The key idea of **TorchDynamo**: it does not try to understand all of Python. It runs your function under a bytecode interpreter; PyTorch operations are recorded into the graph, while ordinary Python (loops, `if` on Python values, function calls) is simply *executed* during tracing and disappears from the graph. A `for` loop over 6 layers therefore becomes 6 copies of the layer in the graph.

We can watch Dynamo at work by giving `torch.compile` a **custom backend**: a function that receives the captured FX graph. Ours just prints it and returns it unchanged:

In [ ]:
def show_graph_backend(gm: torch.fx.GraphModule, example_inputs):
    print("=== graph captured by Dynamo (shown as Python code) ===")
    print(gm.code)
    return gm.forward          # run the graph as-is (no optimization)

def f(x, w):
    h = x @ w
    for _ in range(2):         # a Python loop: unrolled during tracing
        h = torch.relu(h * 2 + 1)
    return h.sum()

f_compiled = torch.compile(f, backend=show_graph_backend)
x, w = torch.randn(4, 8), torch.randn(8, 3)
print(f_compiled(x, w), f(x, w))

The loop is gone: the graph simply contains `relu(mul(...))` twice. With the default backend (`"inductor"`), this graph would now be fused into one or two GPU kernels.

## 3. Guards and recompilation, dynamic shapes

A compiled graph is only valid under the assumptions made while tracing it: the input shapes, dtypes and devices, the values of Python variables used in `if` statements, whether the model is in `train()` or `eval()` mode, and so on. Dynamo records these assumptions as **guards** and checks them (cheaply) at every call:

* guards pass → reuse the cached compiled code;
* a guard fails → **recompile** for the new situation, and cache that too.

Shapes are the most common cause of recompilation. By default, the first call is compiled for the exact shapes seen; if a second call has a different size, Dynamo recompiles once more with that dimension marked **dynamic** (symbolic), so that later sizes reuse the dynamic version. `TORCH_LOGS="recompiles"` (or `torch._logging.set_logs(recompiles=True)` in a notebook) explains every recompilation:

In [ ]:
torch._dynamo.reset()                        # forget earlier compilations
torch._logging.set_logs(recompiles=True)

@torch.compile(backend="eager")              # "eager" backend: capture graphs but don't optimize (fast to demo)
def scale(x):
    return x * 2

scale(torch.randn(4))
scale(torch.randn(4))       # same shape: guards pass, no recompile
scale(torch.randn(8))       # new size: recompiles, marking the size as dynamic
scale(torch.randn(16))      # handled by the dynamic graph
scale(torch.randn(16, dtype=torch.float64))   # new dtype: recompiles again
torch._logging.set_logs(recompiles=False)

Things to know about shapes:

* `torch.compile(model, dynamic=False)` always specializes on exact shapes (fastest code, but one compilation per shape); `dynamic=True` makes shapes symbolic from the start; the default (`None`) starts static and becomes dynamic when a shape changes.
* `torch._dynamo.mark_dynamic(x, 0)` declares in advance that dimension 0 of `x` will vary.
* Variable-length inputs (text!) cause many shapes. Common fixes: pad to a fixed length or to a few "bucket" lengths, and drop the last incomplete batch. This matters even more on TPUs (06), where every new shape triggers an XLA compilation.
* There is a limit on recompilations per function (`torch._dynamo.config.recompile_limit`, 8 by default). After that Dynamo gives up and runs the function in eager mode, often silently slowing things down. `TORCH_LOGS="recompiles"` is how you notice.

## 4. Graph breaks

When Dynamo meets something it cannot put into a graph, it does not fail. It performs a **graph break**: it compiles the part captured so far, runs the problematic code in normal Python, and starts a new graph afterwards. Your code still works, but every break means separate small graphs, less fusion, and a trip back to Python.

Common causes:

* turning a tensor into a Python value: `.item()`, `.tolist()`, `if tensor > 0:` (data-dependent control flow);
* calls into code Dynamo cannot trace: some third-party libraries, C extensions, NumPy on tensors in some cases;
* operations with data-dependent output shapes, such as `torch.nonzero` or boolean masking `x[x > 0]`.

`torch._dynamo.explain` reports how many graphs and graph breaks a function produces, and why:

In [ ]:
torch._dynamo.reset()

def with_breaks(x):
    y = x.sin()
    if y.sum() > 0:                 # Python `if` on a tensor value -> graph break
        y = y * 2
    scale = y.abs().max().item()    # tensor -> Python number -> graph break
    return y.cos() / scale

explanation = torch._dynamo.explain(with_breaks)(torch.randn(10))
print("graphs:", explanation.graph_count, " graph breaks:", explanation.graph_break_count)
for reason in explanation.break_reasons:
    print(" -", reason.reason.split("\n")[0][:150])

(`generic_jump TensorVariable()` is Dynamo's way of saying "a Python `if` whose condition is a tensor": Python needs a real `True`/`False`, which only exists after the GPU has computed it.)

How to fix graph breaks:

* replace data-dependent Python `if` with tensor operations: `torch.where(cond, a, b)`, or `torch.cond` for real branches;
* move logging and `.item()` calls out of the compiled region (e.g. log every N steps, outside the compiled model);
* compile with `fullgraph=True` to turn every graph break into an **error**, which tells you exactly where they are. Useful for the hot inner part of a model.

In [ ]:
def without_breaks(x):
    y = x.sin()
    y = torch.where(y.sum() > 0, y * 2, y)     # the same logic, as tensor operations
    return y.cos() / y.abs().max()             # keep the scale as a tensor

explanation = torch._dynamo.explain(without_breaks)(torch.randn(10))
print("graphs:", explanation.graph_count, " graph breaks:", explanation.graph_break_count)
print(torch.allclose(with_breaks(torch.ones(3)), without_breaks(torch.ones(3))))

## 5. Compile options

```python
torch.compile(model,
              backend="inductor",      # the compiler; "tpu" on TPUs, "eager"/"aot_eager" for debugging
              mode=None,               # see below
              fullgraph=False,         # True: graph breaks become errors
              dynamic=None)            # see section 3
```

`mode`:

* `None` / `"default"`: a good balance of compile time and speed.
* `"reduce-overhead"`: also uses **CUDA Graphs**, which record a whole sequence of GPU kernel launches once and replay it with a single launch. This removes most of the launch overhead and helps a lot for small batches and for LLM inference (generating one token at a time). It needs static shapes and uses some extra memory.
* `"max-autotune"`: benchmarks several implementations of each matrix multiplication and convolution (and generated Triton kernels) and keeps the fastest. Compilation takes much longer; worth it for long training runs or deployed models. (On a T4 it may warn that there are not enough SMs for some variants, as we saw in 03.)

Practical points:

* **Compilation takes time** (seconds to minutes). The first call, and every recompilation, pays it; the speed-up only shows afterwards. That is why we always ignore the first epoch when measuring.
* **Measure** (03): `torch.compile` gave no speed-up for ResNet9 on a T4, because cuDNN convolutions were already the bottleneck. It helps most when there are many small elementwise operations to fuse (normalization, activations, attention variants, custom losses) or a lot of Python overhead.
* **Regional compilation**: a Transformer is the same block repeated N times. Compiling one block (`for block in model.blocks: block.compile()`) instead of the whole model gives almost the same speed with a fraction of the compile time.
* **Caching**: compiled kernels are cached on disk (`/tmp/torchinductor_<user>`), so a second run in the same environment compiles much faster. `torch.compiler.save_cache_artifacts()` / `load_cache_artifacts()` move that cache to another machine.
* The model returned by `torch.compile` shares its parameters with the original, so you can save `model.state_dict()` as usual.
* 🤗 Transformers: `model.generation_config.cache_implementation = "static"` plus `model.forward = torch.compile(model.forward, mode="reduce-overhead")` is the recipe for fast generation; the `Trainer` has a `torch_compile=True` option.

## 6. `torch.export`: capturing a model for deployment

`torch.compile` is a **just-in-time** compiler: it works inside your Python program and falls back to Python whenever needed. To run a model somewhere else (a server without Python, a phone, another runtime), you need the **whole** model as a graph, captured **ahead of time**, with no graph breaks. That is `torch.export`:

In [ ]:
class SmallNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1, self.fc2 = nn.Linear(8, 16), nn.Linear(16, 2)
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

net = SmallNet().eval()
batch = torch.export.Dim("batch")                      # declare the batch size as dynamic
ep = torch.export.export(net, (torch.randn(4, 8),), dynamic_shapes={"x": {0: batch}})
print(ep.graph_module.code)
print(ep.module()(torch.randn(10, 8)).shape)           # the exported program runs on other batch sizes

An `ExportedProgram` is a pure graph of standard ATen operations plus the weights: no Python code from your model is needed to run it. It can be saved with `torch.export.save`, compiled into a standalone library with **AOTInductor**, or converted for mobile and edge devices with **ExecuTorch**. Unlike `torch.compile`, export never falls back: anything it cannot capture (a graph break) is an error you have to fix.

## 7. `torch.func`: function transforms

`torch.func` (inspired by JAX) treats a model as a pure function of its parameters and offers **transforms** that turn functions into new functions:

* `grad(f)`: a function that returns the gradient of `f`;
* `vmap(f)`: "vectorizing map", runs `f` on every element of a batch, as if you wrote a loop, but as one batched computation;
* `jacrev`, `jacfwd`, `hessian`: Jacobians and Hessians;
* `functional_call(model, params, inputs)`: call a module with a given set of parameters.

A classic use: **per-sample gradients** (the gradient for each example separately, needed for differential privacy and some data-analysis methods). Normal `backward()` only gives the sum over the batch; `vmap(grad(...))` gives all of them in one call:

In [ ]:
from torch.func import functional_call, grad, vmap

model = nn.Linear(5, 1)
params = {k: v.detach() for k, v in model.named_parameters()}
X, y = torch.randn(32, 5), torch.randn(32, 1)

def loss_one(params, x, y):               # the loss of ONE example
    pred = functional_call(model, params, (x[None],))
    return F.mse_loss(pred, y[None])

per_sample = vmap(grad(loss_one), in_dims=(None, 0, 0))(params, X, y)
print({k: tuple(v.shape) for k, v in per_sample.items()})        # one gradient per example
# check: their average is the usual batch gradient
F.mse_loss(model(X), y).backward()
print(torch.allclose(per_sample["weight"].mean(0), model.weight.grad, atol=1e-6))

## 8. Saving memory: activation checkpointing

During training, every layer's outputs (activations) are kept in memory until the backward pass needs them. For deep networks and long sequences, **activations**, not weights, usually fill the GPU.

**Activation checkpointing** (also called gradient checkpointing) keeps only the inputs of selected blocks and **recomputes** their insides during the backward pass. You trade extra compute (typically ~20–30% more time) for much less memory: often the difference between "out of memory" and a model that fits on a T4. In 🤗 Transformers it is `model.gradient_checkpointing_enable()` (TRL's `SFTConfig` turns it on by default, see 06-transformers/05).

In [ ]:
from torch.utils.checkpoint import checkpoint

class Deep(nn.Module):
    def __init__(self, width=2048, depth=16, use_checkpoint=False):
        super().__init__()
        self.layers = nn.ModuleList([nn.Sequential(nn.Linear(width, width), nn.GELU()) for _ in range(depth)])
        self.use_checkpoint = use_checkpoint
    def forward(self, x):
        for layer in self.layers:
            x = checkpoint(layer, x, use_reentrant=False) if self.use_checkpoint else layer(x)
        return x

if device == "cuda":
    for use_checkpoint in [False, True]:
        model = Deep(use_checkpoint=use_checkpoint).cuda()
        x = torch.randn(4096, 2048, device="cuda", requires_grad=True)
        torch.cuda.reset_peak_memory_stats()
        model(x).sum().backward()
        print(f"checkpointing={use_checkpoint}: peak memory {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
        del model, x
else:
    print("(this memory demo needs a CUDA GPU)")

## 9. Finding the bottleneck: `torch.profiler`

Before optimizing anything, **measure where the time goes**. `torch.profiler` records every operation and kernel, on the CPU and on the GPU:

In [ ]:
from torch.profiler import profile, ProfilerActivity

model = nn.Sequential(nn.Linear(1024, 4096), nn.GELU(), nn.Linear(4096, 1024)).to(device)
x = torch.randn(256, 1024, device=device)
activities = [ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if device == "cuda" else [])
with profile(activities=activities) as prof:
    for _ in range(5):
        model(x).sum().backward()
sort_by = "cuda_time_total" if device == "cuda" else "cpu_time_total"
print(prof.key_averages().table(sort_by=sort_by, row_limit=8))

What to look for:

* which operations take the most time (here the matrix multiplications, as expected);
* gaps where the GPU is idle: the CPU is the bottleneck (Python overhead, data loading as in 03, or synchronization such as `.item()` every step);
* many tiny kernels: candidates for fusion with `torch.compile`.

`prof.export_chrome_trace("trace.json")` writes a timeline you can open in [Perfetto](https://ui.perfetto.dev) to see exactly when each kernel ran.

## 10. FlexAttention: custom attention, compiled

Attention variants keep appearing: causal masks, sliding windows, ALiBi or relative position biases, document masks for packed sequences, ... Writing a fast fused kernel for each one by hand (like FlashAttention) is hard. **FlexAttention** lets you write the variant as a tiny Python function that modifies one attention score, and uses `torch.compile` to generate a fused kernel for it:

```python
def score_mod(score, b, h, q_idx, kv_idx):     # called (conceptually) for every score
    return score + bias[h] * (kv_idx - q_idx)   # e.g. an ALiBi-style distance bias
```

For masks, a `mask_mod` returns whether a position may be attended to, and `create_block_mask` precomputes which blocks are completely masked, so they are skipped entirely.

In [ ]:
from torch.nn.attention.flex_attention import flex_attention, create_block_mask

def causal(b, h, q_idx, kv_idx):
    return q_idx >= kv_idx

def sliding_window(b, h, q_idx, kv_idx):          # causal, but only the last 64 tokens
    return (q_idx >= kv_idx) & (q_idx - kv_idx < 64)

B, H, T, D = 2, 4, 1024, 64
q, k, v = torch.randn(3, B, H, T, D, device=device).unbind(0)
try:
    flex = torch.compile(flex_attention)
    for name, mask_mod in [("causal", causal), ("sliding window", sliding_window)]:
        block_mask = create_block_mask(mask_mod, B=None, H=None, Q_LEN=T, KV_LEN=T, device=device)
        out = flex(q, k, v, block_mask=block_mask)
        print(f"{name:15s} output {tuple(out.shape)}, blocks computed: {block_mask.sparsity():.0f}% sparse")
    # the causal version should match the built-in causal attention
    ref = F.scaled_dot_product_attention(q, k, v, is_causal=True)
    out = flex(q, k, v, block_mask=create_block_mask(causal, None, None, T, T, device=device))
    print("matches SDPA:", torch.allclose(out, ref, atol=1e-3))
except Exception as e:
    print("FlexAttention is not available on this device:", type(e).__name__, str(e)[:200])

## Summary

| Tool | What it is for |
|---|---|
| `torch.compile` | speed: capture graphs at run time, fuse and optimize them, fall back to Python when needed |
| guards, `dynamic=`, `TORCH_LOGS="recompiles"` | understanding and avoiding recompilation |
| `torch._dynamo.explain`, `fullgraph=True` | finding graph breaks |
| `mode="reduce-overhead"` / `"max-autotune"` | CUDA Graphs for small batches; autotuning for maximum speed |
| `torch.export` | the whole model as a graph, ahead of time, for deployment (AOTInductor, ExecuTorch) |
| `torch.func` | function transforms: per-sample gradients, Jacobians, vectorization |
| activation checkpointing | trade compute for memory |
| `torch.profiler` | find out where the time actually goes |
| FlexAttention | custom attention variants with fused-kernel speed |

## Further reading

* [Introduction to `torch.compile`](https://docs.pytorch.org/tutorials/intermediate/torch_compile_tutorial.html) and the [`torch.compiler` documentation](https://docs.pytorch.org/docs/stable/torch.compiler.html), including the troubleshooting guide
* [Dynamo deep dive](https://docs.pytorch.org/docs/stable/torch.compiler_dynamo_deepdive.html): guards, graph breaks, how bytecode tracing works
* [`torch.export`](https://docs.pytorch.org/docs/stable/export.html) and [AOTInductor](https://docs.pytorch.org/docs/stable/torch.compiler_aot_inductor.html)
* [`torch.func`](https://docs.pytorch.org/docs/stable/func.html), including per-sample gradients
* [Activation checkpointing (`torch.utils.checkpoint`)](https://docs.pytorch.org/docs/stable/checkpoint.html)
* [PyTorch Profiler tutorial](https://docs.pytorch.org/tutorials/recipes/recipes/profiler_recipe.html)
* [FlexAttention blog post](https://pytorch.org/blog/flexattention/) and [API](https://docs.pytorch.org/docs/stable/nn.attention.flex_attention.html)